# 06 · Troubleshooting — symptom → fix

**Audience:** anyone whose reconstruction is wrong, slow, or won't
start. This notebook is organised by **symptom**. Skim the headings,
jump to your problem.

Each section is self-contained — it builds a small synthetic dataset
that exhibits the failure, then shows the fix.

Sections:

1. [Reconstruction is all black / NaN](#1-reconstruction-is-all-black--nan)
2. [Ring artefacts persist after cleanup tuning](#2-ring-artefacts-persist-after-cleanup-tuning)
3. [Arc / comet-tail artefacts](#3-arc--comet-tail-artefacts)
4. [Reconstruction is upside down or mirrored](#4-reconstruction-is-upside-down-or-mirrored)
5. [Output cube is the wrong shape](#5-output-cube-is-the-wrong-shape)
6. [Memory / OMP errors](#6-memory--omp-errors)
7. [Reconstruction is unusually slow](#7-reconstruction-is-unusually-slow)
8. [Cleanup tuning picks the baseline (config 0)](#8-cleanup-tuning-picks-the-baseline-config-0)


In [1]:
# --- Setup ---------------------------------------------------------
%matplotlib inline
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Make the MIDAS TOMO Python API importable
MIDAS_TOMO = os.path.expanduser('~/opt/MIDAS/TOMO')
if MIDAS_TOMO not in sys.path:
    sys.path.insert(0, MIDAS_TOMO)
NB_DIR = os.path.join(MIDAS_TOMO, 'notebooks')
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

import _phantom as phantom

# Working directory for this notebook's outputs
WORK = os.path.expanduser('~/tomo_notebooks/06_troubleshoot')
os.makedirs(WORK, exist_ok=True)
print('Working dir:', WORK)


Working dir: /Users/hsharma/tomo_notebooks/06_troubleshoot


## 1 · Reconstruction is all black / NaN

**Symptom:** every slice is uniform zero, NaN, or near-zero noise.

**Most common cause:** the white field is at or below the dark level,
so `(data - dark) / (bright - dark)` underflows. Always check that
`mean(bright) >> mean(dark)`.


In [2]:
vol = phantom.make_phantom(48, 48, 48)
angles = np.arange(0, 180, 2.0, dtype=np.float32)
acq_ok = phantom.make_acquisition(vol, angles, add_rings=False,
                                  white_level=20000, dark_level=100)

# Sabotage: white at dark level
bad_whites = np.full_like(acq_ok.whites_before, 100.0)
print('mean(bright):', bad_whites.mean(),
      '   mean(dark):', acq_ok.dark.mean())
print('=> bright - dark ≈ 0  =>  normalization will collapse')


mean(bright): 100.0    mean(dark): 99.96704
=> bright - dark ≈ 0  =>  normalization will collapse


In [3]:
from midas_tomo_python import run_tomo
whites_ok = np.stack([acq_ok.whites_before.mean(0),
                      acq_ok.whites_after.mean(0)])
whites_bad = np.stack([bad_whites.mean(0), bad_whites.mean(0)])

r_ok = run_tomo(acq_ok.projections, acq_ok.dark, whites_ok,
                WORK, angles, shifts=0.0, numCPUs=4, doCleanup=1)
r_bad = run_tomo(acq_ok.projections, acq_ok.dark, whites_bad,
                 WORK, angles, shifts=0.0, numCPUs=4, doCleanup=1)
print('OK   std:', r_ok.std(), '   range:', r_ok.min(), r_ok.max())
print('BAD  std:', r_bad.std(), '   range:', r_bad.min(), r_bad.max())


Time elapsed in preprocessing: 0.001s.
Version: MIDAS v11.0 (8a155821)
Total number of thetas: 90
We are doing all slices. Total number of slices: 48

          MIDAS TOMO - Configuration Summary
  Data Input:
    Data File       : /Users/hsharma/tomo_notebooks/06_troubleshoot/input.bin
    Input Type      : Raw Projections
    Recon Output    : /Users/hsharma/tomo_notebooks/06_troubleshoot/output
  Detector:
    Dimensions      : 48 x 48 (X x Y)
  Angles:
    Theta Count     : 90
    Range           : 0.00 to 178.00
  Reconstruction:
    Filter          : 2
    Shift Range     : 0.00 to 0.00 (step 1.00, n=1)
    Slices          : 48
    Auto Centering  : Yes
    Log Projection  : Yes
    Extra Padding   : 0
    Save Separate   : No
  Corrections:
    Ring Removal    : No
    Stripe Removal  : No

Sinograms are not a power of 2.  They will be increased to 64
Reading wisdom file fftwf_wisdom_2d_128.txt.
Memory needed per process: 1097802, Total available RAM: 68719476736, MaxNProcs: 625

**Fix.** Plot the dark and bright means as a quick sanity-check before
every reconstruction:

```python
print('dark mean   :', dark.mean())
print('bright mean :', whites.mean())
print('ratio       :', whites.mean() / dark.mean(), '  (should be ≥ 10)')
```

If the ratio is < 10, check the beamline log — most likely the shutter
was closed during the white-field acquisition.


## 2 · Ring artefacts persist after cleanup tuning

**Symptom:** you ran the cleanup sweep, picked the best config, and
still see rings.

**Most common causes:**

a) **The sweep grid did not include aggressive enough configs.** The
   built-in grid scales windows up to `detector_width / 3`. For very
   broad rings you may need `detector_width / 2`.

b) **The stripes are too narrow for `sm_size`.** If `sm_size` is e.g.
   11, you cannot correct a 5-px-wide stripe. Try `sm_size = 5`.

c) **The artefact is not a Vo-class stripe.** Some rings come from
   sample motion or beam instability — those need different
   correction (or none).

Below: a custom grid that goes wider than the default.


In [4]:
from midas_tomo_python import run_tomo_cleanup_sweep

vol = phantom.make_phantom(64, 64, 64)
angles = np.arange(0, 180, 1.0, dtype=np.float32)
acq = phantom.make_acquisition(vol, angles, add_rings=True)
whites = np.stack([acq.whites_before.mean(0), acq.whites_after.mean(0)])

aggressive_grid = [
    {'snr': 0.0, 'la': 0,  'sm': 0},
    {'snr': 3.0, 'la': 31, 'sm': 11},    # default-ish
    {'snr': 1.0, 'la': 41, 'sm': 15},    # more sensitive + broader
    {'snr': 1.0, 'la': 51, 'sm': 17},    # even broader
    {'snr': 0.8, 'la': 31, 'sm': 7},     # narrow but very sensitive
]
res = run_tomo_cleanup_sweep(acq.projections, acq.dark, whites,
                             WORK, angles, shift=0.0,
                             cleanup_configs=aggressive_grid,
                             numCPUs=4, doCleanup=0)
print('best:', res['best_config'])


Version: MIDAS v11.0 (8a155821)
Total number of thetas: 180
We are reading the slices file: /Users/hsharma/tomo_notebooks/06_troubleshoot/cleanup_tuning_slices.txt.

          MIDAS TOMO - Configuration Summary
  Data Input:
    Data File       : /Users/hsharma/tomo_notebooks/06_troubleshoot/cleanup_tuning_input.bin
    Input Type      : Raw Projections
    Recon Output    : /Users/hsharma/tomo_notebooks/06_troubleshoot/cleanup_tuning_output
  Detector:
    Dimensions      : 64 x 64 (X x Y)
  Angles:
    Theta Count     : 180
    Range           : 0.00 to 179.00
  Reconstruction:
    Filter          : 2
    Shift Range     : 0.00 to 0.10 (step 0.10, n=2)
    Slices          : 4
    Auto Centering  : Yes
    Log Projection  : Yes
    Extra Padding   : 0
    Save Separate   : No
  Corrections:
    Ring Removal    : No
    Stripe Removal  : Yes SWEEP over 5 configs (file: /Usecleanup-sweep MIDAS_TOMO took 0.07s
rs/hsharma/tomo_notebooks/06_troubleshoot/cleanup_tuning_grid.txt)

  Cleanup 

Saved /Users/hsharma/tomo_notebooks/06_troubleshoot/cleanup_tuning_montage.png
best: {'snr': 1.0, 'la': 41, 'sm': 15}


## 3 · Arc / comet-tail artefacts

**Symptom:** crescent or comma shapes around the rotation centre. The
sample is "blurred outward".

**Cause:** rotation-axis shift is wrong. Not by 0.1 px — by 1+ px.

**Fix.** See [04_shift_search.ipynb](04_shift_search.ipynb). Quick
diagnostic: run a coarse shift sweep ±detector_width / 8 and look at
the montage — the right shift produces a recognisable sample.

If your HDF5 file *says* it knows the shift but the reconstruction
looks like an arc anyway, the stored value is stale (common when
samples are re-mounted). Override with `--shifts START END STEP` on
`process_hdf.py`.


## 4 · Reconstruction is upside down or mirrored

**Symptom:** sample features are recognisable but on the wrong side
(left↔right or top↔bottom).

**Cause:** the projection orientation differs from MIDAS's convention.
The HDF5 file may carry a `RotationAngle` to compensate, but if not,
you need to flip / rotate the projections yourself.

**Fix.** Either rotate the projections by 180° in-plane before
feeding them in, or set
`/analysis/process/analysis_parameters/RotationAngle` in the HDF5 to
the angle that `process_hdf.py` should apply.


## 5 · Output cube is the wrong shape

**Symptom:** `np.fromfile(...).reshape(...)` errors out with a
`cannot reshape array of size N` message.

**Causes:**

a) **Wrong `xDimNew`.** The output is padded to the next power of 2
   above `detXdim`. If `ExtraPad 1`, it is doubled again.

b) **`saveReconSeparate=1`** writes one file per (slice, shift) pair
   instead of one big binary. Check for `*_slice_NNNNN_shift_NNN_*` files.

c) **`stripeConfigFile`** was set, so the filename now has a
   `NrCleanup_NNN_` prefix and the cube has an extra dimension.
   Companion file `<basename>_cleanup_configs.txt` lists each
   cleanup config in cube order.

The filename always tells you the exact shape — read each numeric
field:


In [5]:
fn = ('recon_NrCleanup_004_NrShifts_002_NrSlices_00004_'
      'XDim_000128_YDim_000128_float32.bin')
import re
m = re.findall(r'_(\w+?)_0*(\d+)', fn)
print(dict(m))


{'NrCleanup': '4', 'NrShifts': '2', 'NrSlices': '4', 'XDim': '128', 'YDim': '128'}


## 6 · Memory / OMP errors

**Symptom:** segfault, `bad_alloc`, or "Cannot allocate memory".

**Cause:** `MIDAS_TOMO` reads `/proc/meminfo` and self-limits its
thread count, but the limit is conservative — on shared machines
something else may have eaten the headroom by the time it runs.

**Fix.** Reduce `nCPUs` (CLI arg / `numCPUs` kwarg). Memory cost is
roughly `(2 × xDimNew^2) bytes × n_threads` per FFT buffer × small
factor.


## 7 · Reconstruction is unusually slow

**Symptom:** a 2k × 2k × 1500 dataset takes much longer than ~2 min.

**Things to check:**

- **First-run wisdom.** FFTW wisdom files are generated on first use
  for each detector size and cached as
  `fftwf_wisdom_{1d,2d}_NNNN.txt` in the working dir. Subsequent
  runs are typically 2-3× faster. Don't delete them.
- **GPU build.** If MIDAS is built with `-DUSE_CUDA=ON`, there is a
  `MIDAS_TOMO_GPU` binary that uses a multi-pair batched gridrec.
  Pass `useGPU=True` to `run_tomo()`. Cleanup sweep is currently
  CPU-only.
- **Sinograms not power-of-2.** If `xDim` is far from the next power
  of 2 (e.g. 1100 padded to 2048), you are wasting half the work.
  Consider cropping to a power of 2 if you don't need the full FOV.


## 8 · Cleanup tuning picks the baseline (config 0)

**Symptom:** `run_tomo_cleanup_sweep` returns `best_config = {'snr': 0,
'la': 0, 'sm': 0}`. The auto-picker thinks "do nothing" is best.

**Diagnosis paths:**

a) **There really aren't significant rings.** Look at the montage. If
   all panels look similar, congratulations — your data is already
   clean.

b) **The ring metric is fooled by real features.** Samples with
   concentric structure (battery jelly-rolls, capillary walls) can
   drive the metric up even after stripe removal. In this case the
   sweep should *still* pick a cleaned config because the auto-pick
   has a tie-breaker that prefers cleaned configs within 1 % of
   baseline. If it doesn't, your rings are weak — keep the
   baseline.

c) **The dark/white normalisation already removed the stripes.** Some
   detectors have built-in flat-field correction. If your data has
   been pre-corrected, there is nothing left for Vo to do.


## Beyond these eight

If you hit something not covered here, the most informative thing to
inspect first is **the sinogram** (notebook 01, §4). Most failures are
visible there before the recon is even attempted.

For C-engine errors, run `MIDAS_TOMO` directly with `-d 1` (debug
mode) — it dumps intermediate buffers as `.bin` files in the working
directory.
